
# 練習問題: 1つのループで平均と分散を求める (2変数の reduction)

## 目標

`reduction` 節に**複数の変数**を並べることで, 1つのループの中で2つの総和を同時に集計できることを体験する. ここでは配列の和 `s` と二乗和 `sq` を同時に求め, 平均と分散を計算する.

## 課題

`mean_var.cpp` (または `mean_var.f90`) は, 要素数 `n = 1000000` の配列 `a` (`a[i] = sin(i)`) について, 1つのループで

- 和    `s  += a[i]`
- 二乗和 `sq += a[i]*a[i]`

を求め, ループ後に `mean = s/n`, `variance = sq/n - mean*mean` を計算する.

コメント `TODO` の指示に従ってループを並列化せよ. **2つの総和を1つの `reduction` 節にまとめて**指定する.

- C++: ループの直前に `#pragma omp parallel for reduction(+:s,sq)` を加える.
- Fortran: `do` ループを `!$omp parallel do private(x) reduction(+:s,sq)` と `!$omp end parallel do` で囲む (一時変数 `x` はスレッドごとに別々にするため `private` にする).

`reduction(+:s,sq)` のように, カンマで区切って複数の変数を1つの節に並べられる.

## コンパイルと実行

```
# C++
nvc++ -fast -mp=multicore mean_var.cpp -o mean_var.exe

# Fortran
nvfortran -fast -mp=multicore mean_var.f90 -o mean_var.exe
```

```
OMP_NUM_THREADS=4 ./mean_var.exe
```

## 期待される結果

平均はほぼ 0, 分散はほぼ 0.5 になる (`sin` の値の性質による). 例:

```
mean = 0.000000, variance = 0.499999
```

`OMP_NUM_THREADS` を変えても結果はほぼ同じになる. ただし浮動小数点の和は**足す順序**によってごくわずかに変わるため, スレッド数を変えると最下位の桁が変動しうることに注意せよ.



# 1. ツールの読み込み
- AIチュータ及びジョブ投入ツールの読み込み (カーネル起動後に一度実行すればよい)
  - `heytutor` : `%%hey` でAIチュータに質問できるようになる (使い方は末尾を参照)
  - `wisteria_submit` : `%%bash_submit` (先頭に `#PJM ...` を書く) でジョブ投入できるようになる


In [ ]:
import heytutor
import wisteria_submit

# 2. C++ ベースコード

In [ ]:
%%writefile_ mean_var.cpp
#include <cstdio>
#include <cmath>

int main() {
  const long n = 1000000L;
  static double a[n];
  for (long i = 0; i < n; i++) {
    a[i] = sin((double)i);
  }
  double s = 0.0, sq = 0.0;
  // TODO: 下のループを #pragma omp parallel for reduction(+:s,sq) で並列化し, 2つの総和の競合を解消せよ.
  for (long i = 0; i < n; i++) {
    double x = a[i];
    s  += x;
    sq += x * x;
  }
  double mean = s / n;
  double var  = sq / n - mean * mean;
  printf("mean = %f, variance = %f\n", mean, var);
  return 0;
}

## 2-1. コンパイル

In [ ]:
%%bash_
PATH=/work/opt/local/x86_64/cores/nvidia/23.3/Linux_x86_64/23.3/compilers/bin:/opt/FJSVxtclanga/tcsds-1.2.41/bin:$PATH
nvc++ -fast -mp=multicore mean_var.cpp -o mean_var_cpp.exe

## 2-2. 実行
- 計算ノードにジョブとして投入して実行する。スレッド数・キュー・制限時間は `#PJM` 行で調整する。
- すぐにログインノードで試したいときは, 先頭の `%%bash_submit` を `%%bash_` に書き換えて (`#PJM` 行はコメントなので無視される) 実行すればよい。ただし数秒で終わる軽いジョブに限る。

In [ ]:
%%bash_submit
#PJM -L rscgrp=lecture-a
#PJM -L elapse=0:01:00
#PJM -L gpu=1
#PJM --omp thread=9
#PJM -g gt69
#PJM -j
#PJM -o 0output.txt

./mean_var_cpp.exe

# 3. Fortran ベースコード

In [ ]:
%%writefile_ mean_var.f90
program mean_var
  integer(8), parameter :: n = 1000000_8
  real(8), allocatable :: a(:)
  real(8) :: s, sq, x, mean, var
  integer(8) :: i
  allocate(a(0:n-1))
  do i = 0, n - 1
     a(i) = sin(real(i, 8))
  end do
  s = 0.0d0
  sq = 0.0d0
  ! TODO: 下のループを !$omp parallel do private(x) reduction(+:s,sq) で並列化し, 2つの総和の競合を解消せよ.
  do i = 0, n - 1
     x = a(i)
     s  = s + x
     sq = sq + x * x
  end do
  ! TODO: 上で始めた parallel do 領域を閉じる (!$omp end parallel do).
  mean = s / n
  var  = sq / n - mean * mean
  print "(a,f0.6,a,f0.6)", "mean = ", mean, ", variance = ", var
end program mean_var

## 3-1. コンパイル

In [ ]:
%%bash_
PATH=/work/opt/local/x86_64/cores/nvidia/23.3/Linux_x86_64/23.3/compilers/bin:/opt/FJSVxtclanga/tcsds-1.2.41/bin:$PATH
nvfortran -fast -mp=multicore mean_var.f90 -o mean_var_f90.exe

## 3-2. 実行
- 計算ノードにジョブとして投入して実行する。スレッド数・キュー・制限時間は `#PJM` 行で調整する。
- すぐにログインノードで試したいときは, 先頭の `%%bash_submit` を `%%bash_` に書き換えて (`#PJM` 行はコメントなので無視される) 実行すればよい。ただし数秒で終わる軽いジョブに限る。

In [ ]:
%%bash_submit
#PJM -L rscgrp=lecture-a
#PJM -L elapse=0:01:00
#PJM -L gpu=1
#PJM --omp thread=9
#PJM -g gt69
#PJM -j
#PJM -o 0output.txt

./mean_var_f90.exe


# 4. AIチュータへの質問の仕方 (参考)
- 先頭で `import heytutor` 済みなら, セルに `%%hey` と書いて質問できる。
- ChatGPTなどと同様に自由に質問してよい。ただし「答えをそのまま教えて」などは自制すること。
- セル内で使える変数 (自動で展開される):
  - `{file:FILENAME}` : _FILENAME_ の中身 (例: `{file:problem.md}`, `{file:mean_var.cpp}`)
  - `{bash[-1]}` : 最後に実行した `%%bash_` セルの入力・出力, `{bash[-2]}` = その前, ...
- 以下は質問例 (必要に応じてコピーして使う。Fortranなら `.cpp` を `.f90` に書き換える)。

## 4-1. 一般的な質問

In [ ]:
%%hey

C++の関数定義の文法どんなだっけ?

## 4-2. この問題に関するヒント

In [ ]:
%%hey

この問題に関するヒントを教えて

問題:
{file:problem.md}

## 4-3. 困ったときのヘルプ
- コンパイル時や実行時のエラー直後に実行するとエラーに関するヘルプが得られる。

In [ ]:
%%hey

以下のエラーが出た。何が間違い?

プログラム:
{file:mean_var.cpp}

コマンドと実行結果:
{bash[-1]}

## 4-4. フィードバック
- 答えが出た後も, 無駄なところやより良いやり方がないかを聞くことを推奨。

In [ ]:
%%hey

私の答に対するフィードバックをください。

問題:
{file:problem.md}

私の答:
{file:mean_var.cpp}